In [2]:
import pandas as pd
import os

In [33]:
OUTPUT_DIR = "pipeline_results"

In [37]:
DIRS = {
    'mmseqs_results': os.path.join(OUTPUT_DIR, '01_mmseqs_results'),
    'paired_proteins': os.path.join(OUTPUT_DIR, '02_paired_proteins'),
    'nucleotides': os.path.join(OUTPUT_DIR, '03_nucleotides'),
    'alignments': os.path.join(OUTPUT_DIR, '04_alignments'),
    'pal2nal': os.path.join(OUTPUT_DIR, '05_pal2nal'),
    'trees': os.path.join(OUTPUT_DIR, '06_trees'),
    'labeled_trees': os.path.join(OUTPUT_DIR, '07_labeled_trees'),
    'busted': os.path.join(OUTPUT_DIR, '08_busted'),
    'relax': os.path.join(OUTPUT_DIR, '08b_relax'),
    'busted_background': os.path.join(OUTPUT_DIR, '08c_busted_background'),
    'absrel': os.path.join(OUTPUT_DIR, '09_absrel'),
    'meme': os.path.join(OUTPUT_DIR, '09b_meme'),
    'fitmg94': os.path.join(OUTPUT_DIR, '10_fitmg94'),
    'summary': os.path.join(OUTPUT_DIR, '11_summary'),
    'tmp': os.path.join(OUTPUT_DIR, 'tmp'),
}
for dir_name, dir_path in DIRS.items():
    os.makedirs(dir_path, exist_ok=True)
    print(f"Created: {dir_path}")

print("\nDirectory structure created!")

Created: pipeline_results/01_mmseqs_results
Created: pipeline_results/02_paired_proteins
Created: pipeline_results/03_nucleotides
Created: pipeline_results/04_alignments
Created: pipeline_results/05_pal2nal
Created: pipeline_results/06_trees
Created: pipeline_results/07_labeled_trees
Created: pipeline_results/08_busted
Created: pipeline_results/08b_relax
Created: pipeline_results/08c_busted_background
Created: pipeline_results/09_absrel
Created: pipeline_results/09b_meme
Created: pipeline_results/10_fitmg94
Created: pipeline_results/11_summary
Created: pipeline_results/tmp

Directory structure created!


# 1. Mmseq2

на вход: 
- объединенные и протэганные (название вида перед > (Род_вид)) птичьи протеомы 
- человеческие белки--таргеты
выход: таблица квери--таргет
потом из нее оставляем по лучшему хиту на вид

чтобы сделать по категориям захожу в окружение где стоит mmseq(based) и запускаю цикл (файл mmseq_cicl.sh, не забыть сделать исполняемым)по всем файлам из папки со следующей следующими параметрами PARAMS="-c 0.2 --min-seq-id 0.3 --alt-ali 25 --threads 12"


результаты лежат в папке dn_ds_pipeline/pipeline_results/01_mmseqs_results/category

### обработка результатов
1. собираем файлы по кусочкам + добавляем столбец с тегом+добавляем LQ и то находится ли LQ в вехних 20 процентах
2. ищем лучшее вхождение для данного вида
3. оставляем только гены, для которых достаточное количество видов имеют гомологи + где есть хотя бы 5 долгоживущих
4. долгоживущие определяются как верхние 20 процентов

In [1]:
import os
import pandas as pd
df=pd.DataFrame()
folder = "pipeline_results/01_mmseqs_results/category"
for filename in os.listdir(folder):
    filepath = os.path.join(folder, filename)
    if os.path.isfile(filepath):          # только файлы, не папки
        # делаем что-то с файлом
        columns = ["qseqid", "sseqid", "pident", "length", "mismatch", "gapopen",
               "qstart", "qend", "sstart", "send", "evalue", "bitscore"]
    
        df_mmseqs = pd.read_csv(filepath, sep='\t', names=columns)
        teg=filepath.split('/')[-1][10:-3]
        df_mmseqs['teg']=teg
        df_mmseqs['tag'] = df_mmseqs['teg'].astype(str).str.replace(r'_+', ' ', regex=True)
        
        
        df = pd.concat([df, df_mmseqs], axis=0)
        df['tag']=df['tag'].str.split('.').str[0]
        print(filepath.split('/')[-1][10:-2])
df


Age-related_changes_in_gene_expression__methylation_or_protein_activity__genes_proteins.fasta.
Age-related_changes_in_gene_expression__methylation_or_protein_activity_in_non-mammals__genes_proteins.
Association_of_the_gene_with_accelerated_aging_in_humans__genes_proteins.
Changes_in_gene_activity_extend_mammalian_lifespan__genes_proteins.
Age-related_changes_in_gene_expression__methylation_or_protein_activity_in_humans__genes_proteins.
Changes_in_gene_activity_protect_against_age-related_impairment__genes_proteins.
Regulation_of_genes_associated_with_aging__genes_proteins.
Changes_in_gene_activity_reduce_mammalian_lifespan__genes_proteins.
Association_of_genetic_variants_and_gene_expression_levels_with_longevity__genes_proteins.
uncat_genes_proteins.
Changes_in_gene_activity_enhance_age-related_deterioration__genes_proteins.
Changes_in_gene_activity_extend_non-mammalian_lifespan__genes_proteins.


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,teg,tag
0,ITGA3,Buteo_buteo|XP_074892467.1,0.646,712,251,0,3,714,17,726,2.592000e-310,964,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...
1,ITGA3,Buteo_buteo|XP_074892466.1,0.646,712,251,0,3,714,17,726,2.592000e-310,964,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...
2,ITGA3,Aquila_chrysaetos_chrysaetos|XP_029877306.1,0.646,712,252,0,3,714,16,727,2.592000e-310,964,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...
3,ITGA3,Buteo_buteo|XP_074892465.1,0.646,712,251,0,3,714,17,726,2.592000e-310,964,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...
4,ITGA3,Buteo_buteo|XP_074892464.1,0.646,712,251,0,3,714,17,726,2.592000e-310,964,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38,TYMS,Falco_cherrug|XP_055561646.1,0.895,162,17,0,25,186,141,302,5.031000e-99,323,Changes_in_gene_activity_extend_non-mammalian_...,Changes in gene activity extend non-mammalian ...
39,TYMS,Pelecanus_crispus|XP_075561712.1,0.895,163,17,0,24,186,140,302,5.031000e-99,323,Changes_in_gene_activity_extend_non-mammalian_...,Changes in gene activity extend non-mammalian ...
40,TYMS,Calonectris_borealis|XP_074998125.1,0.895,162,17,0,25,186,141,302,5.031000e-99,323,Changes_in_gene_activity_extend_non-mammalian_...,Changes in gene activity extend non-mammalian ...
41,SMOC2,Cuculus_canorus|XP_053919303.1,0.898,445,45,0,8,446,2,446,6.169000e-279,857,Changes_in_gene_activity_extend_non-mammalian_...,Changes in gene activity extend non-mammalian ...


In [2]:

# Examine MMseqs2 results

    
print(f"Total hits: {len(df)}")
print(f"Unique query genes: {df['qseqid'].nunique()}")
print(f"Unique subject sequences: {df['sseqid'].nunique()}")
print(f"\nFirst 10 rows:")
display(df.head(10))


Total hits: 9501
Unique query genes: 999
Unique subject sequences: 9144

First 10 rows:


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,teg,tag
0,ITGA3,Buteo_buteo|XP_074892467.1,0.646,712,251,0,3,714,17,726,2.592000e-310,964,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...
1,ITGA3,Buteo_buteo|XP_074892466.1,0.646,712,251,0,3,714,17,726,2.592000e-310,964,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...
2,ITGA3,Aquila_chrysaetos_chrysaetos|XP_029877306.1,0.646,712,252,0,3,714,16,727,2.592000e-310,964,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...
3,ITGA3,Buteo_buteo|XP_074892465.1,0.646,712,251,0,3,714,17,726,2.592000e-310,964,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...
4,ITGA3,Buteo_buteo|XP_074892464.1,0.646,712,251,0,3,714,17,726,2.592000e-310,964,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...
5,ITGA3,Buteo_buteo|XP_074892463.1,0.646,712,251,0,3,714,17,726,2.592000e-310,964,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...
6,ITGAE,Harpia_harpyja|XP_052659040.1,0.496,970,478,0,185,1133,175,1144,2.239000e-291,930,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...
7,ITGAE,Harpia_harpyja|XP_052659037.1,0.496,970,478,0,185,1133,395,1364,2.239000e-291,930,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...
8,ITGAE,Harpia_harpyja|XP_052659036.1,0.496,970,478,0,185,1133,436,1405,2.239000e-291,930,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...
9,FZD3,Caloenas_nicobarica|XP_065487238.1,0.944,468,26,0,1,468,1,468,2.224000e-313,959,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...


In [3]:
df['spices']=df['sseqid'].str.split('|').str[0]

In [4]:
# =============================================================================
# STEP 2: EXTRACT BEST MATCHES
# =============================================================================

columns = ["qseqid", "sseqid", "pident", "length", "mismatch", "gapopen",
           "qstart", "qend", "sstart", "send", "evalue", "bitscore"]

df['spices']=df['sseqid'].str.split('|').str[0]
print(f"Loaded {len(df)} hits from MMseqs2 results")

best_matches = pd.DataFrame()

for prefix in set(df['spices']):
    bat_matches = df[df['sseqid'].str.startswith(prefix)]
    if not bat_matches.empty:
        best_bat_matches = bat_matches.loc[bat_matches.groupby('qseqid')['pident'].idxmax()]
        best_matches = pd.concat([best_matches, best_bat_matches], ignore_index=True)

# Save best matches
BEST_MATCHES_CSV = os.path.join('pipeline_results/01_mmseqs_results', 'best_matches.csv')
best_matches.to_csv(BEST_MATCHES_CSV, sep='\t', index=False)

print(f"\nFound {len(best_matches)} best matches")
print(f"Unique genes: {best_matches['qseqid'].nunique()}")
print(f"\nBest matches saved to: {BEST_MATCHES_CSV}")

# Summary statistics
bat_counts = best_matches.groupby('qseqid')['spices'].nunique()
print(f"\nBirds species per gene:")
print(f"  Median: {bat_counts.median():.0f}")
print(f"  Min: {bat_counts.min()}")
print(f"  Max: {bat_counts.max()}")
print(f"\nMedian percent identity: {best_matches['pident'].median():.2f}%")

Loaded 9501 hits from MMseqs2 results

Found 5640 best matches
Unique genes: 999

Best matches saved to: pipeline_results/01_mmseqs_results/best_matches.csv

Birds species per gene:
  Median: 1
  Min: 1
  Max: 107

Median percent identity: 0.96%


In [5]:
##оставляем только гомологи с достаточным количеством видов которые захитились
good_genes = best_matches.groupby('qseqid')['sseqid'].count()
good_genes = good_genes[good_genes > 30].index

best_matches_filtered = best_matches[best_matches['qseqid'].isin(good_genes)]

In [6]:
best_matches_filtered

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,teg,tag,spices
0,ACTA1,Motacilla_alba_alba|XP_037989541.1,1.000,377,0,0,1,377,1,377,4.984000e-255,784,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Motacilla_alba_alba
1,ACTG1,Motacilla_alba_alba|XP_038013158.1,1.000,375,0,0,1,375,1,375,1.213000e-252,777,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Motacilla_alba_alba
2,AHCYL2,Motacilla_alba_alba|XP_038010124.1,1.000,404,0,0,1,404,100,503,1.931000e-279,856,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Motacilla_alba_alba
3,ARPP19,Motacilla_alba_alba|XP_038003243.1,0.973,112,3,0,1,112,1,112,7.292000e-67,226,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Motacilla_alba_alba
5,CALM1,Motacilla_alba_alba|XP_037988483.1,1.000,113,0,0,1,113,37,149,2.821000e-69,233,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Motacilla_alba_alba
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5632,SRP54,Catharus_ustulatus|XP_032918911.1,0.987,400,5,0,1,400,105,504,6.996000e-263,808,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Catharus_ustulatus
5633,SRSF6,Catharus_ustulatus|XP_032930908.1,0.932,89,6,0,1,89,1,89,1.053000e-49,178,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Catharus_ustulatus
5637,UBL3,Catharus_ustulatus|XP_032907570.1,0.990,109,1,0,1,109,9,117,2.562000e-68,230,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Catharus_ustulatus
5638,WNT7B,Catharus_ustulatus|XP_032914826.1,0.934,353,23,0,1,353,1,353,2.080000e-235,726,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Catharus_ustulatus


In [7]:
best_matches_filtered = best_matches_filtered.copy()  # if it was a slice

In [8]:
best_matches_filtered['spices'] = best_matches_filtered['spices'].apply(lambda x: x.split('_')[:2] if isinstance(x, str) else x)

In [9]:
best_matches_filtered

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,teg,tag,spices
0,ACTA1,Motacilla_alba_alba|XP_037989541.1,1.000,377,0,0,1,377,1,377,4.984000e-255,784,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,"[Motacilla, alba]"
1,ACTG1,Motacilla_alba_alba|XP_038013158.1,1.000,375,0,0,1,375,1,375,1.213000e-252,777,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,"[Motacilla, alba]"
2,AHCYL2,Motacilla_alba_alba|XP_038010124.1,1.000,404,0,0,1,404,100,503,1.931000e-279,856,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,"[Motacilla, alba]"
3,ARPP19,Motacilla_alba_alba|XP_038003243.1,0.973,112,3,0,1,112,1,112,7.292000e-67,226,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,"[Motacilla, alba]"
5,CALM1,Motacilla_alba_alba|XP_037988483.1,1.000,113,0,0,1,113,37,149,2.821000e-69,233,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,"[Motacilla, alba]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5632,SRP54,Catharus_ustulatus|XP_032918911.1,0.987,400,5,0,1,400,105,504,6.996000e-263,808,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,"[Catharus, ustulatus]"
5633,SRSF6,Catharus_ustulatus|XP_032930908.1,0.932,89,6,0,1,89,1,89,1.053000e-49,178,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,"[Catharus, ustulatus]"
5637,UBL3,Catharus_ustulatus|XP_032907570.1,0.990,109,1,0,1,109,9,117,2.562000e-68,230,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,"[Catharus, ustulatus]"
5638,WNT7B,Catharus_ustulatus|XP_032914826.1,0.934,353,23,0,1,353,1,353,2.080000e-235,726,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,"[Catharus, ustulatus]"


In [10]:
best_matches_filtered['spices'] = best_matches_filtered['spices'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)

In [11]:

ll=pd.read_csv('featured_and_annotated_birds.csv')
ll=ll[ll['refseq_anootated']==1]
threshold = ll['LQ'].quantile(0.8)

# 2. Создаём новый столбец с булевой меткой (True для долгоживущих)
ll['upper_20_persent_LQ'] = ll['LQ'] >= threshold
ll['spices']=ll['Genus']+' '+ll['Species']

In [12]:
ll[['spices', 'LQ', 'upper_20_persent_LQ']]

,spices,LQ,upper_20_persent_LQ
0,Anas platyrhynchos,1.266340,False
1,Taeniopygia guttata,1.214389,False
2,Gallus gallus,1.380465,False
3,Coturnix japonica,0.396286,False
4,Anser cygnoides,1.075807,False
...,...,...,...
112,Centrocercus urophasianus,0.271465,False
113,Empidonax traillii,1.102982,False
117,Corvus brachyrhynchos,1.051600,False
119,Fulmarus glacialis,2.280308,True


In [13]:
result = pd.merge(best_matches_filtered, ll[['spices', 'LQ', 'upper_20_persent_LQ']], on='spices', how='left')
result

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,teg,tag,spices,LQ,upper_20_persent_LQ
0,ACTA1,Motacilla_alba_alba|XP_037989541.1,1.000,377,0,0,1,377,1,377,4.984000e-255,784,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Motacilla alba,1.215399,False
1,ACTG1,Motacilla_alba_alba|XP_038013158.1,1.000,375,0,0,1,375,1,375,1.213000e-252,777,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Motacilla alba,1.215399,False
2,AHCYL2,Motacilla_alba_alba|XP_038010124.1,1.000,404,0,0,1,404,100,503,1.931000e-279,856,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Motacilla alba,1.215399,False
3,ARPP19,Motacilla_alba_alba|XP_038003243.1,0.973,112,3,0,1,112,1,112,7.292000e-67,226,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Motacilla alba,1.215399,False
4,CALM1,Motacilla_alba_alba|XP_037988483.1,1.000,113,0,0,1,113,37,149,2.821000e-69,233,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Motacilla alba,1.215399,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3548,SRP54,Catharus_ustulatus|XP_032918911.1,0.987,400,5,0,1,400,105,504,6.996000e-263,808,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Catharus ustulatus,0.949260,False
3549,SRSF6,Catharus_ustulatus|XP_032930908.1,0.932,89,6,0,1,89,1,89,1.053000e-49,178,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Catharus ustulatus,0.949260,False
3550,UBL3,Catharus_ustulatus|XP_032907570.1,0.990,109,1,0,1,109,9,117,2.562000e-68,230,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Catharus ustulatus,0.949260,False
3551,WNT7B,Catharus_ustulatus|XP_032914826.1,0.934,353,23,0,1,353,1,353,2.080000e-235,726,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Catharus ustulatus,0.949260,False


In [14]:
# Считаем количество long-lived записей на каждый seqid
long_lived_counts = result[result['upper_20_persent_LQ'] == True].groupby('qseqid').size()

# Отбираем seqid с >=5
valid_seqid = long_lived_counts[long_lived_counts >= 5].index

# Фильтруем исходный датафрейм
df_filtered = result[result['qseqid'].isin(valid_seqid)]


In [15]:
f=result.groupby('qseqid')['upper_20_persent_LQ'].sum()
d=result.groupby('qseqid').count()['sseqid']
finale=pd.merge(f, d, on='qseqid')
finale=pd.merge(finale, result[['qseqid', 'tag']].drop_duplicates(), on='qseqid')

In [36]:
result.to_csv('homologs_filtered.csv')

In [24]:
finale


,qseqid,upper_20_persent_LQ,sseqid,tag
0,ACTA1,19,94,Age-related changes in gene expression methyla...
1,ACTG1,18,79,Age-related changes in gene expression methyla...
2,AHCYL2,17,85,Age-related changes in gene expression methyla...
3,ARPP19,9,51,Age-related changes in gene expression methyla...
4,CALM1,22,107,Age-related changes in gene expression methyla...
5,CALM3,22,107,Age-related changes in gene expression methyla...
6,CDC42,22,104,Age-related changes in gene expression methyla...
7,CELF2,9,72,Age-related changes in gene expression methyla...
8,CIRBP,7,32,Age-related changes in gene expression methyla...
9,CTNNA1,9,34,Age-related changes in gene expression methyla...


## делаем красивые таблички с результатами поиска гомологов латеховские и для презы

In [17]:
tab = finale.rename(columns={
    'upper_20_persent_LQ': 'number of species long-lived',
    'sseqid': 'number of species', 
    'qseqid':'gene', 
    'tag':'hallmark'

})
styled_df = tab.style.background_gradient(cmap='Blues', axis=None)


In [18]:
styled_df.to_html('ttable.html')


In [19]:
import pandas as pd
import matplotlib.pyplot as plt

# Получаем LaTeX-код таблицы
latex_code = tab.to_latex()

# Удаляем столбец с индексом из вывода
lines = latex_code.split('\n')
# Находим строку с \begin{tabular}
start = next(i for i, line in enumerate(lines) if line.startswith('\\begin{tabular}'))
# Убираем первый символ в спецификации столбцов (он обычно 'l')
if start + 1 < len(lines):
    lines[start + 1] = lines[start + 1][1:]  # убираем 'l' в начале
# В строке заголовков убираем первый столбец
if start + 2 < len(lines) and '&' in lines[start + 2]:
    header_parts = lines[start + 2].split('&')
    lines[start + 2] = '&'.join(header_parts[1:])
# В каждой строке данных убираем первый столбец
for i in range(start + 3, len(lines)):
    if '&' in lines[i]:
        parts = lines[i].split('&')
        lines[i] = '&'.join(parts[1:])

# Сохраняем исправленный LaTeX-код
with open('table.tex', 'w') as f:
    f.write('\n'.join(lines))

# Генерация PNG через matplotlib
fig, ax = plt.subplots(figsize=(12, 0.5 + len(tab) * 0.2))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=tab.values, colLabels=tab.columns, loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(9)
plt.savefig('table.png', dpi=200, bbox_inches='tight')
plt.close()

print("Сохранено: table.tex и table.png")

Сохранено: table.tex и table.png


In [22]:
import pandas as pd
import re

def escape_latex(s):
    """Экранирует спецсимволы LaTeX в строке."""
    if not isinstance(s, str):
        s = str(s)
    replacements = [
        ('\\', r'\textbackslash{}'),
        ('_', r'\_'),
        ('%', r'\%'),
        ('&', r'\&'),
        ('#', r'\#'),
        ('$', r'\$'),
        ('{', r'\{'),
        ('}', r'\}'),
        ('~', r'\textasciitilde{}'),
        ('^', r'\textasciicircum{}'),
    ]
    for old, new in replacements:
        s = s.replace(old, new)
    return s

def df_to_latex_full_document(df, filename, caption, label):
    """Сохраняет DataFrame в файл LaTeX, который представляет собой полный документ с таблицей."""
    # Определяем выравнивание столбцов
    col_align = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            col_align.append('r')
        else:
            col_align.append('l')
    col_format = ''.join(col_align)

    # Экранируем заголовки и данные
    headers = [escape_latex(col) for col in df.columns]
    data = df.applymap(escape_latex).values

    # Строим содержимое таблицы
    table_lines = []
    table_lines.append(r'\begin{table}[htbp]')
    table_lines.append(r'\centering')
    table_lines.append(r'\caption{' + caption + '}')
    table_lines.append(r'\label{' + label + '}')
    table_lines.append(r'\begin{adjustbox}{width=\textwidth}')
    table_lines.append(r'\begin{tabular}{' + col_format + '}')
    table_lines.append(r'\toprule')
    table_lines.append(' & '.join(headers) + r' \\')
    table_lines.append(r'\midrule')
    for row in data:
        table_lines.append(' & '.join(row) + r' \\')
    table_lines.append(r'\bottomrule')
    table_lines.append(r'\end{tabular}')
    table_lines.append(r'\end{adjustbox}')
    table_lines.append(r'\end{table}')

    # Полный документ с преамбулой
    preamble = [
        r'\documentclass{article}',
        r'\usepackage[utf8]{inputenc}',
        r'\usepackage{geometry}',
        r'\geometry{a4paper, margin=1cm}',
        r'\usepackage{booktabs}',
        r'\usepackage{adjustbox}',
        r'\begin{document}'
    ]
    postamble = [r'\end{document}']

    all_lines = preamble + table_lines + postamble

    with open(filename, 'w', encoding='utf-8') as f:
        f.write('\n'.join(all_lines))

# Пример использования для вашего DataFrame `tab`
df_to_latex_full_document(
    df=tab,
    filename='table.tex',
    caption='Results of homology search using mmseqs reciprocal best hit',
    label='tab:homology_results'
)
print("Готово! Полный LaTeX-документ сохранён в table.tex")

Готово! Полный LaTeX-документ сохранён в table.tex


/tmp/ipykernel_2467607/1528787965.py:37: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  data = df.applymap(escape_latex).values


In [28]:
df_filtered

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,teg,tag,spices,LQ,upper_20_persent_LQ
0,ACTA1,Motacilla_alba_alba|XP_037989541.1,1.000,377,0,0,1,377,1,377,4.984000e-255,784,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Motacilla alba,1.215399,False
1,ACTG1,Motacilla_alba_alba|XP_038013158.1,1.000,375,0,0,1,375,1,375,1.213000e-252,777,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Motacilla alba,1.215399,False
2,AHCYL2,Motacilla_alba_alba|XP_038010124.1,1.000,404,0,0,1,404,100,503,1.931000e-279,856,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Motacilla alba,1.215399,False
3,ARPP19,Motacilla_alba_alba|XP_038003243.1,0.973,112,3,0,1,112,1,112,7.292000e-67,226,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Motacilla alba,1.215399,False
4,CALM1,Motacilla_alba_alba|XP_037988483.1,1.000,113,0,0,1,113,37,149,2.821000e-69,233,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Motacilla alba,1.215399,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3548,SRP54,Catharus_ustulatus|XP_032918911.1,0.987,400,5,0,1,400,105,504,6.996000e-263,808,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Catharus ustulatus,0.949260,False
3549,SRSF6,Catharus_ustulatus|XP_032930908.1,0.932,89,6,0,1,89,1,89,1.053000e-49,178,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Catharus ustulatus,0.949260,False
3550,UBL3,Catharus_ustulatus|XP_032907570.1,0.990,109,1,0,1,109,9,117,2.562000e-68,230,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Catharus ustulatus,0.949260,False
3551,WNT7B,Catharus_ustulatus|XP_032914826.1,0.934,353,23,0,1,353,1,353,2.080000e-235,726,Age-related_changes_in_gene_expression__methyl...,Age-related changes in gene expression methyla...,Catharus ustulatus,0.949260,False


In [29]:
# Для каждого уникального tag подсчитываем количество уникальных qseqid
result.groupby('tag')['qseqid'].nunique().reset_index(name='unique_qseqid_count')

,tag,unique_qseqid_count
0,Age-related changes in gene expression methyla...,2
1,Age-related changes in gene expression methyla...,38
2,Age-related changes in gene expression methyla...,1
3,Association of genetic variants and gene expre...,2
4,Changes in gene activity extend mammalian life...,2
5,Changes in gene activity reduce mammalian life...,1
6,Regulation of genes associated with aging gene...,1


In [37]:
# Группировка по tag: считаем уникальные qseqid и среднее pident
hallmarks = result.groupby('tag').agg(
    unique_qseqid_count=('qseqid', 'nunique'),
    mean_pident=('pident', 'mean')
).reset_index()

hallmarks['mean_pident'] = hallmarks['mean_pident'].round(2)

In [38]:
tab = hallmarks.rename(columns={
    'unique_qseqid_count':'Number of genes gene', 
    'tag':'Hallmark',
    'mean_pident':'Mean percent of identity '

})

In [39]:
import pandas as pd
import re

def escape_latex(s):
    """Экранирует спецсимволы LaTeX в строке."""
    if not isinstance(s, str):
        s = str(s)
    replacements = [
        ('\\', r'\textbackslash{}'),
        ('_', r'\_'),
        ('%', r'\%'),
        ('&', r'\&'),
        ('#', r'\#'),
        ('$', r'\$'),
        ('{', r'\{'),
        ('}', r'\}'),
        ('~', r'\textasciitilde{}'),
        ('^', r'\textasciicircum{}'),
    ]
    for old, new in replacements:
        s = s.replace(old, new)
    return s

def df_to_latex_full_document(df, filename, caption, label):
    """Сохраняет DataFrame в файл LaTeX, который представляет собой полный документ с таблицей."""
    # Определяем выравнивание столбцов
    col_align = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            col_align.append('r')
        else:
            col_align.append('l')
    col_format = ''.join(col_align)

    # Экранируем заголовки и данные
    headers = [escape_latex(col) for col in df.columns]
    data = df.applymap(escape_latex).values

    # Строим содержимое таблицы
    table_lines = []
    table_lines.append(r'\begin{table}[htbp]')
    table_lines.append(r'\centering')
    table_lines.append(r'\caption{' + caption + '}')
    table_lines.append(r'\label{' + label + '}')
    table_lines.append(r'\begin{adjustbox}{width=\textwidth}')
    table_lines.append(r'\begin{tabular}{' + col_format + '}')
    table_lines.append(r'\toprule')
    table_lines.append(' & '.join(headers) + r' \\')
    table_lines.append(r'\midrule')
    for row in data:
        table_lines.append(' & '.join(row) + r' \\')
    table_lines.append(r'\bottomrule')
    table_lines.append(r'\end{tabular}')
    table_lines.append(r'\end{adjustbox}')
    table_lines.append(r'\end{table}')

    # Полный документ с преамбулой
    preamble = [
        r'\documentclass{article}',
        r'\usepackage[utf8]{inputenc}',
        r'\usepackage{geometry}',
        r'\geometry{a4paper, margin=1cm}',
        r'\usepackage{booktabs}',
        r'\usepackage{adjustbox}',
        r'\begin{document}'
    ]
    postamble = [r'\end{document}']

    all_lines = preamble + table_lines + postamble

    with open(filename, 'w', encoding='utf-8') as f:
        f.write('\n'.join(all_lines))

# Пример использования для вашего DataFrame `tab`
df_to_latex_full_document(
    df=tab,
    filename='hallmarks_homology_rbh.tex',
    caption='Open Genes: Hallmarks of aging associated genes',
    label='tab:homology_results'
)
print("Готово! Полный LaTeX-документ сохранён в table.tex")

Готово! Полный LaTeX-документ сохранён в table.tex


/tmp/ipykernel_2467607/2115496056.py:37: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  data = df.applymap(escape_latex).values
